# Inverse design quickstart

This notebook will get users up and running with a very simple inverse design optimization with `tidy3d`. Inverse design uses the "adjoint method" to compute gradients of a figure of merit with respect to design parameters using only 2 simulations no matter how many design parameters are present. This gradient is then used to do high dimensional, gradient-based optimization of the system.

The setup we'll demonstrate here involves a point dipole source and a point field monitor on either side of a dielectric box. Using the adjoint plugin in `tidy3d`, we use gradient-based optimization to maximize the intensity enhancement at the measurement spot with respect to the box size in all 3 dimensions.

<img src="img/Adjoint_Quickstart.png" width="300" alt="Schematic of the design problem.">

For more detailed notebooks, see these

* [Tidy3D Autograd Tutorial](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd1Intro/).

* [Topology Optimization](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd3InverseDesign/).

* [Shape Optimization](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd5BoundaryGradients/).

* [Grating Coupler Inverse Design](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd6GratingCoupler/).


In [1]:
# To install other packages needed, uncomment lines below.
# !pip install optax

In [ ]:
import tidy3d as td
from tidy3d.web import run
import matplotlib.pylab as plt

import autograd as ag
import autograd.numpy as anp
import optax
from tidy3d.web.core.environment import Env, dev, prod
import tidy3d.web as web

Env.set_current(dev)
web.configure("TIDY3D_DEV_API_KEY")

## Setup

First, we set up some basic parameters and "static" components of our simulation.

In [3]:
# wavelength and frequency
wavelength = 1.55
freq0 = td.C_0 / wavelength

# permittivity of box
eps_box = 2

# size of sim in x,y,z
L = 10 * wavelength

# spc between sources, monitors, and PML / box
buffer = 1.0 * wavelength

In [4]:
# create a source to the left of sim
source = td.PointDipole(
    center=(-L / 2 + buffer, 0, 0),
    source_time=td.GaussianPulse(freq0=freq0, fwidth=freq0 / 10.0),
    polarization="Ez",
)

In [5]:
# create a monitor to right of sim for measuring intensity
monitor = td.FieldMonitor(
    center=(+L / 2 - buffer, 0, 0),
    size=(0.0, 0.0, 0.0),
    freqs=[freq0],
    name="point",
)

In [6]:
# create "base" simulation (the box will be added inside of the objective function later)
sim = td.Simulation(
    size=(L, L, L),
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=25),
    structures=[],
    sources=[source],
    monitors=[monitor],
    run_time=120 / freq0,
)

## Define objective function

Now we construct our objective function out of some helper functions. Our objective function measures the intensity enhancement at the measurement point as a function of a design parameter that controls the box size.

In [7]:
# function to get box size (um) as a function of the design parameter (-inf, inf)

size_min = 0
size_max = L - 4 * buffer

trans_min, trans_max = -L / 4, L / 4


def get_size(param: float):
    """Size of box as function of parameter, smoothly maps (-inf, inf) to (size_min, size_max)."""
    param_01 = 0.5 * (anp.tanh(param) + 1)
    return (size_max * param_01) + (size_min * (1 - param_01))

def get_translation(param_translate: float):
    """Maps a parameter to translation in x-direction."""
    param_01 = 0.5 * (anp.tanh(param_translate) + 1)
    return trans_min + (trans_max - trans_min) * param_01

# Transformed Geometry

In [8]:
def make_sim(param_size: float, param_translate: float):
    """Constructs simulation with box size and translation."""

    if param_size is None:
        return sim.copy()

    size_box = get_size(param_size)
    center_box = get_translation(param_translate)
    
    box = td.Structure(
        geometry=td.Box(center=(0., 0., 0.), size=(1., 1., 1.)).scaled(x=size_box, y=size_box, z=size_box).translated(x=center_box, y = 0., z = 0.),
        medium=td.Medium(permittivity=eps_box),
    )
    return sim.updated_copy(structures=[box])

# Box Geometry

In [9]:
# def make_sim(param_size: float, param_translate: float):
#     """Constructs simulation with box size and translation."""

#     if param_size is None:
#         return sim.copy()

#     size_box = get_size(param_size)
#     center_box = (get_translation(param_translate), get_translation(param_translate), get_translation(param_translate))
    
#     box = td.Structure(
#         geometry=td.Box(center=center_box, size=(size_box, size_box, size_box)),
#         medium=td.Medium(permittivity=eps_box),
#     )
#     return sim.updated_copy(structures=[box])

In [10]:
# function to compute and measure intensity as function of the design paramater


def measure_intensity(sim_data: td.SimulationData) -> float:
    """get intensity from SimulationData."""
    return anp.sum(sim_data.get_intensity(monitor.name).values)


def intensity(param_size: float, param_translate: float) -> float:
    """Intensity measured at monitor as function of parameter."""

    # make the sim using the parameter value
    sim_with_square = make_sim(param_size, param_translate)

    # run sim through tidy3d web API
    data = run(sim_with_square, task_name="inverse_design", verbose=False, local_gradient=True)

    # evaluate the intensity at the measurement position
    return measure_intensity(data)

In [ ]:
# get the intensity with no box, for normalization (care about enhancement, not abs value)
intensity_norm = intensity(param_size=None, param_translate=None)
print(f"With no box, intensity = {intensity_norm:.4f}.")
print("This value will be used for normalization of the objective function.")

In [12]:
def objective_fn(params):
    """Objective function: maximize intensity at monitor."""
    return intensity(params[0], params[1]) / intensity_norm

## Optimization Loop

Next, we use `autograd` to construct a function that returns the gradient of our objective function and use this to run our gradient-based optimization in a for loop.

In [13]:
# use autograd to get function that returns objective function and its gradient
val_and_grad_fn = ag.value_and_grad(objective_fn)

In [ ]:
# hyperparameters
num_steps = 4
learning_rate = 0.05

# initialize adam optimizer with starting parameter
params = anp.array([ -0.5, 0.0])
optimizer = optax.adam(learning_rate=learning_rate)
opt_state = optimizer.init(params)

# store history
objective_history = []  # the normalized objective function with no box
param_history = [params.copy()]  # -100 is approximately "no box" (size=0)

for i in range(num_steps):
    # compute gradient and current objective funciton value
    value, gradient = val_and_grad_fn(params)
    gradient = anp.array(gradient, dtype=float)
    # compute and apply updates to the optimizer based on gradient (-1 sign to maximize obj_fn)
    updates, opt_state = optimizer.update(-gradient, opt_state, params)
    params = optax.apply_updates(params, updates)
    params = anp.array(params, dtype=float)
    objective_history.append(value)
    param_history.append(params.copy())

    # outputs
    print(f"Step {i+1}: size={get_size(params[0]):.4f}, trans={get_translation(params[1]):.4f}, intensity={value:.4f}")

## Analysis
Finally we plot our results: optimization progress, field pattern, and box size vs intensity enhancement.

In [ ]:
# objective function vs iteration number
plt.plot(objective_history)
plt.xlabel("iteration number")
plt.ylabel("intensity enhancement (unitless)")
plt.title("intensity enhancement during optimization")
plt.show()

In [16]:
# construct simulation with final parameters
sim_final = make_sim(param_size=param_history[-1][0], param_translate=param_history[-1][1])

# add a field monitor for plotting
fld_mnt = td.FieldMonitor(
    center=(+L / 2 - buffer, 0, 0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    name="fields",
)
sim_final = sim_final.updated_copy(monitors=[monitor, fld_mnt])

# run simulation
data_final = run(sim_final, task_name="quickstart_final", verbose=False)

In [17]:
# record final intensity
intensity_final = measure_intensity(data_final)
intensity_final_normalized = intensity_final / intensity_norm

objective_history.append(intensity_final_normalized)

In [ ]:
# plot intensity distribution
ax = data_final.plot_field(
    field_monitor_name="fields", field_name="E", val="abs^2", vmax=intensity_final
)

ax.plot(source.center[0], 0, marker="o", mfc="limegreen", mec="black", ms=10)
ax.plot(monitor.center[0], 0, marker="o", mfc="orange", mec="black", ms=10)
plt.show()

In [ ]:
# scatter the intensity enhancement vs the box size
sizes = [get_size(p) for p in param_history]
objective_history = objective_history
_ = plt.scatter(sizes, objective_history)
ax = plt.gca()
ax.set_xlabel("box size (um)")
ax.set_ylabel("intensity enhancement (unitless)")
plt.title("intensity enhancement vs. box size")
plt.show()